In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/__huggingface_repos__.json
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-5835/config.json
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-5835/trainer_state.json
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-5835/training_args.bin
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-5835/scheduler.pt
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-5835/model.safetensors
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-5835/optimizer.pt
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-5835/rng_state.pth
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-5835/generation_config.json
/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-17505/config.json
/kaggle/input/nllb-hindi-finetuned/pyto

In [2]:
import pandas as pd
#data1 = pd.read_csv('/kaggle/input/whisper-others-output/whisper_small_op.csv')
data1=pd.read_csv('/kaggle/input/english-2-hindi-dev-mappings/en-hi train.csv')
data1

,Start segment,End segment,Audio,Source text,Translated text
0,2.610000,4.940000,mkb-hi28.wav,My dear countrymen Namaskar,"मेरे प्यारे देशवासियो, नमस्कार"
1,6.030000,13.060000,mkb-hi28.wav,When I express Mann Ki Baat it feels like I a...,जब मैं मन की बात करता हूं तो ऐसा लगता है जैसे ...
2,14.070000,27.989999,mkb-hi28.wav,Little matters exchanged that teach one anothe...,छोटी-छोटी बातों का आदान-प्रदान एक-दूसरे को खट्...
3,29.120000,40.180000,mkb-hi28.wav,Today is the last day of January 2021 Are you...,आज जनवरी 2021 का आखिरी दिन है क्या आप भी मेरी ...
4,41.010000,49.949999,mkb-hi28.wav,It just doesnt feel that the entire month of J...,ऐसा महसूस ही नहीं होता कि जनवरी का पूरा महीना ...
...,...,...,...,...,...
205196,1858.090000,1860.060000,hi127108005-20.wav,This is the end of the Module 1,यह मॉड्यूल 1 का अंत है
205197,1860.110000,1870.790000,hi127108005-20.wav,Module 2 presents through 20 units of Instruct...,मॉड्यूल 2 एनबीए द्वारा प्रदान किए गए ढांचे में...
205198,1872.389999,1897.910000,hi127108005-20.wav,So if you want to design and offer a course h...,इसलिए यदि आप कोई पाठ्यक्रम डिजाइन और प्रस्तुत ...
205199,1898.940000,1907.459999,hi127108005-20.wav,Thank you very much for attention and I welcom...,ध्यान देने के लिए आपका बहुत-बहुत धन्यवाद और मै...


In [3]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
import gc  # Garbage collection for memory optimization
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
from torch.utils.data import Dataset

# 🚀 Set device and enable multiple GPUs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs.")

# 🚀 Define language codes for NLLB-200
src_lang, tgt_lang = "eng_Latn", "hin_Deva"
#model_path = "/kaggle/input/nllb-finetuned/pytorch/default/1/nllb_results/checkpoint-15000"
# 🚀 Load Model & Tokenizer for Fine-Tuning
base_model_name = "facebook/nllb-200-distilled-600M"  # Use 3.3B for a larger version
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model_path='/kaggle/input/nllb-hindi-finetuned/pytorch/default/5/nllb_results/checkpoint-15000'
# 🚀 Load Model
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# 🚀 Load Training and Validation Data
train_data_path = '/kaggle/input/english-2-hindi-dev-mappings/en-hi train.csv'
dev_data_path = '/kaggle/input/english-2-hindi-dev-mappings/en-hi dev.csv'

df_train = pd.read_csv(dev_data_path).iloc[:20000, :]
df_dev = pd.read_csv(dev_data_path).iloc[:20000, :]

# ✅ Check if required columns exist
required_columns = {"Source text", "Translated text"}
if not required_columns.issubset(df_train.columns):
    raise ValueError("Training CSV must contain 'Source text' and 'Translated text' columns.")

if not required_columns.issubset(df_dev.columns):
    raise ValueError("Validation CSV must contain 'Source text' and 'Translated text' columns.")

# 🚀 Prepare training input sentences
source_texts = df_train["Source text"].astype(str).tolist()
target_texts = df_train["Translated text"].astype(str).tolist()

# ✅ Remove NaN values (if any)
source_texts = [text if text.strip() != 'nan' else '' for text in source_texts]
target_texts = [text if text.strip() != 'nan' else '' for text in target_texts]

# 🚀 Tokenization with padding & truncation
train_encodings = tokenizer(
    source_texts,
    padding="longest",
    truncation=True,
    return_tensors="pt",
    return_attention_mask=True
)
label_encodings = tokenizer(
    target_texts,
    padding="longest",
    truncation=True,
    return_tensors="pt",
    return_attention_mask=True
)

# ✅ Replace padding tokens with -100 for loss calculation
labels = label_encodings["input_ids"]
labels[labels == tokenizer.pad_token_id] = -100

# 🚀 Prepare validation input sentences
dev_source_texts = df_dev["Source text"].astype(str).tolist()
dev_target_texts = df_dev["Translated text"].astype(str).tolist()

# 🚀 Tokenization for validation set
dev_encodings = tokenizer(
    dev_source_texts,
    padding="longest",
    truncation=True,
    return_tensors="pt",
    return_attention_mask=True
)
dev_label_encodings = tokenizer(
    dev_target_texts,
    padding="longest",
    truncation=True,
    return_tensors="pt",
    return_attention_mask=True
)

# ✅ Ensure `labels` padding tokens are replaced with -100 for validation
dev_labels = dev_label_encodings["input_ids"]
dev_labels[dev_labels == tokenizer.pad_token_id] = -100

# ✅ Create PyTorch Dataset Class
class TranslationDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }

# ✅ Create dataset instances
train_dataset = TranslationDataset(train_encodings, labels)
dev_dataset = TranslationDataset(dev_encodings, dev_labels)

# 🚀 Set Training Arguments
training_args = TrainingArguments(
    output_dir="./nllb_results",
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",  # Saves at the end of each epoch
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",
    disable_tqdm=False, 
)

# 🚀 Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset
)

# 🚀 Start Fine-tuning
trainer.train()

# 🚀 Save Fine-Tuned Model
model.save_pretrained("fine_tuned_nllb200")
tokenizer.save_pretrained("fine_tuned_nllb200")

print("✅ Fine-tuning completed! Model saved at 'fine_tuned_nllb200'.")


'import torch\nimport pandas as pd\nimport gc  # Garbage collection for memory optimization\nfrom tqdm import tqdm\nfrom transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer\nfrom torch.utils.data import Dataset\n\n# 🚀 Set device and enable multiple GPUs\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nif torch.cuda.device_count() > 1:\n    print(f"Using {torch.cuda.device_count()} GPUs.")\n\n# 🚀 Define language codes for NLLB-200\nsrc_lang, tgt_lang = "eng_Latn", "hin_Deva"\n#model_path = "/kaggle/input/nllb-finetuned/pytorch/default/1/nllb_results/checkpoint-15000"\n# 🚀 Load Model & Tokenizer for Fine-Tuning\nbase_model_name = "facebook/nllb-200-distilled-600M"  # Use 3.3B for a larger version\ntokenizer = AutoTokenizer.from_pretrained(base_model_name)\nmodel_path=\'/kaggle/input/nllb-hindi-finetuned/pytorch/default/5/nllb_results/checkpoint-15000\'\n# 🚀 Load Model\nmodel = AutoModelForSeq2SeqLM.from_pretrained(model_path)\n\n# 🚀 Load Training

In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd
import torch

# Paths to model and input file
model_path = "/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-17505"
input_csv = "/kaggle/input/whisper-others-output/whisper_small_op.csv"
output_csv = "/kaggle/working/dev_translated_output.csv"

# Load model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
base_model_name = "facebook/nllb-200-distilled-600M"  # Use the base model for tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Sample translation test
sample_text = "Hello, how are you?"  
inputs = tokenizer(sample_text, return_tensors="pt").to(device)
output_tokens = model.generate(**inputs)
translated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("Translated Text:", translated_text)

# Load input CSV (first 10 rows for quick testing)
df = pd.read_csv(input_csv)#.iloc[:10, :]

# Check if "Predicted Source text" column exists
if "Predicted Source text" not in df.columns:
    raise ValueError("Column 'Predicted Source text' not found in CSV file.")

# Create the "Translated text" column before filling it
df["Predicted Translated text"] = ""

# Function to translate text
def translate(text):
    if pd.isna(text) or text.strip() == "":  # Handle empty or NaN values
        return ""
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        output_tokens = model.generate(**inputs, max_length=256)
    return tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0]

# Apply translation function
df["Predicted Translated text"] = df["Predicted Source text"].apply(translate)

# Save output CSV
df.to_csv(output_csv, index=False)

print(f"Translation complete. Output saved to {output_csv}")


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

Translated Text: नमस्ते, आप कैसे हैं?
Translation complete. Output saved to /kaggle/working/dev_translated_output.csv


# For Test Set

In [6]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from tqdm import tqdm

# Load the dataset
df = pd.read_csv("/kaggle/input/english-to-bengali-test-mapping/en-bn test.csv")

# Extract the source texts
source_texts = df["Source text"].tolist()

# Define the model name and model path
model_name = "facebook/nllb-200-distilled-600M"
model_path = "/kaggle/input/nllb-hindi-finetuned/pytorch/default/6/nllb_results/checkpoint-17505"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Create translation pipeline
translator = pipeline("translation", model=model, tokenizer=tokenizer, 
                      src_lang="eng_Latn", tgt_lang="hin_Deva")

# Translate each text with progress bar
translated_texts = []
for text in tqdm(source_texts, desc="Translating", unit="sentence"):
    translation = translator(text)
    translated_texts.append(translation[0]["translation_text"])

# Append translations to DataFrame
df["Translated Text"] = translated_texts

# Save DataFrame with translations to CSV
df.to_csv("translated_output.csv", index=False)

# Save just the translated texts to a plain text file
with open("translated_texts.txt", "w", encoding="utf-8") as f:
    for line in translated_texts:
        f.write(line + "\n")

print("✅ Translation complete!")
print("📄 CSV saved as 'translated_output.csv'")
print("📄 Text file saved as 'translated_texts.txt'")

Device set to use cuda:0
Translating:  49%|████▉     | 17803/36245 [2:15:51<1:47:21,  2.86sentence/s]Token indices sequence length is longer than the specified maximum sequence length for this model (4032 > 1024). Running this sequence through the model will result in indexing errors
Your input_length: 4032 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Translating: 100%|██████████| 36245/36245 [4:46:47<00:00,  2.11sentence/s]


✅ Translation complete!
📄 CSV saved as 'translated_output.csv'
📄 Text file saved as 'translated_texts.txt'


In [7]:
#df

In [8]:
'''import pandas as pd
df=pd.read_csv('/kaggle/input/english-hindi-model-version3-dev-test-results/dev_translated_output.csv')
df1=pd.read_csv('/kaggle/input/english-2-hindi-dev-mappings/en-hi dev.csv')#.iloc[:10,:]'''

"import pandas as pd\ndf=pd.read_csv('/kaggle/input/english-hindi-model-version3-dev-test-results/dev_translated_output.csv')\ndf1=pd.read_csv('/kaggle/input/english-2-hindi-dev-mappings/en-hi dev.csv')#.iloc[:10,:]"

In [9]:
'''import sacrebleu

# Extract reference translations and generated hypotheses
references = df1.iloc[:, -1].astype(str).tolist()  # Convert to string to avoid NaN issues
hypotheses = df.iloc[:, -2].astype(str).tolist()  

# Filter out empty or NaN translations
hypotheses = [h if h.strip() else "" for h in hypotheses]
references = [[r if r.strip() else ""] for r in references]  # SacreBLEU expects a list of lists

# Calculate BLEU score
bleu = sacrebleu.corpus_bleu(hypotheses, references)

# Print the BLEU score
print(f"SacreBLEU Score: {bleu.score:.2f}")
print('Previous was SacreBLEU Score: 66.87')
'''

'import sacrebleu\n\n# Extract reference translations and generated hypotheses\nreferences = df1.iloc[:, -1].astype(str).tolist()  # Convert to string to avoid NaN issues\nhypotheses = df.iloc[:, -2].astype(str).tolist()  \n\n# Filter out empty or NaN translations\nhypotheses = [h if h.strip() else "" for h in hypotheses]\nreferences = [[r if r.strip() else ""] for r in references]  # SacreBLEU expects a list of lists\n\n# Calculate BLEU score\nbleu = sacrebleu.corpus_bleu(hypotheses, references)\n\n# Print the BLEU score\nprint(f"SacreBLEU Score: {bleu.score:.2f}")\nprint(\'Previous was SacreBLEU Score: 66.87\')\n'

In [10]:
'''import sacrebleu

# Extract reference translations and generated hypotheses
references = df1.iloc[:,-1].astype(str).tolist()  # Fourth column (reference translations)
hypotheses = df.iloc[:, -2].astype(str).tolist()   # Fifth column (generated translations)

# SacreBLEU expects a list of reference lists (even for a single reference)
references = [references]  

# Calculate BLEU score
bleu = sacrebleu.corpus_bleu(hypotheses, references)

# Print the BLEU score
print(f"SacreBLEU Score: {bleu.score:.2f}")
print('Previous was SacreBLEU Score: 31.73')'''

'import sacrebleu\n\n# Extract reference translations and generated hypotheses\nreferences = df1.iloc[:,-1].astype(str).tolist()  # Fourth column (reference translations)\nhypotheses = df.iloc[:, -2].astype(str).tolist()   # Fifth column (generated translations)\n\n# SacreBLEU expects a list of reference lists (even for a single reference)\nreferences = [references]  \n\n# Calculate BLEU score\nbleu = sacrebleu.corpus_bleu(hypotheses, references)\n\n# Print the BLEU score\nprint(f"SacreBLEU Score: {bleu.score:.2f}")\nprint(\'Previous was SacreBLEU Score: 31.73\')'